# 🧼 04 - Basic Cleaning: Missing Values, Types, and Duplicates

This notebook is self-contained — it builds its own messy sample data, so there's nothing to download first. You'll clean a small `df` step by step, then put everything together in a full cleaning **pipeline** on a messier sales dataset.

## ✅ Learning Goals
- Detect and count missing values, and know which "missing" values pandas *can't* see (`""`, `'N/A'`, `-1`)
- Inspect a column's values and types **before** cleaning it
- Fill or drop missing data, and choose between them for a good reason
- Convert columns to numbers, dates, and nullable types safely, and find what failed to convert
- Standardize messy text categories
- Find, inspect, and remove duplicate rows
- Build a cleaning pipeline, and validate the result

> **The core habit:** standardize missing values first, inspect before converting, and validate after every major cleaning step.

## 1. Load a Messy Dataset

In [ ]:
import pandas as pd
import numpy as np

data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', None],
    'Age': ['25', 'thirty', 35, np.nan, '40'],
    'Signup Date': ['2022-01-01', 'not a date', '2022/03/01', None, 'April 5, 2022'],
    'Score': [95.5, None, 88.0, 92.5, ''],
}

df = pd.DataFrame(data)
df

## 2. Detecting Missing or Broken Values

In [ ]:
df.isnull()

In [ ]:
df.isnull().sum()

💡 `.isna()` is exactly the same as `.isnull()`. You'll see both in other people's code.

### What Counts as Missing?
Not every "missing" value looks missing to pandas:

| Value | What it is | Does `.isna()` catch it? |
|---|---|---|
| `np.nan` / `NaN` | Numeric missing value | ✅ Yes |
| `None` | Python's "nothing" | ✅ Yes |
| `pd.NA` | Pandas' missing-value marker | ✅ Yes |
| `""` | An empty string | ❌ No |
| `'N/A'`, `'Unknown'` | Text placeholders | ❌ No, unless you replace them |
| `-1`, `999` | Number placeholders | ❌ No, unless you replace them |

In [ ]:
pd.Series([np.nan, None, pd.NA, '', 'N/A', -1]).isna()

Pandas can't know that `-1` means "unknown" in **your** dataset, so the missing-value count above is only the *visible* missing data. Section 8 turns the hidden kind into real missing values.

In [ ]:
df[df.isnull().any(axis=1)]

Two more counts are handy: the **total** number of missing cells, and the rows where **every** value is missing (often junk rows left over at the end of a file):

In [ ]:
print('Total missing cells:', df.isnull().sum().sum())
df[df.isnull().all(axis=1)]          # no completely empty rows here

### Inspect Before You Clean
Before changing anything, **look** at what's actually in a column. `value_counts(dropna=False)` shows every distinct value, including the missing ones. `.map(type)` shows what **kind** of value each cell holds:

In [ ]:
df['Score'].value_counts(dropna=False)

In [ ]:
df['Age'].map(type).value_counts()

`Age` is a mix of strings (`'25'`, `'thirty'`), an `int`, and a `float` (the `NaN`). A column that holds several types is a sure sign it needs cleaning. `Score` hides an empty string `''` that `.isnull()` didn't count.

## 3. Cleaning Strategy Options

In [ ]:
df.fillna({
    'Name': 'Unknown',
    'Age': -1,
    'Signup Date': '1970-01-01',
    'Score': 0.0,
})

In [ ]:
df.dropna()

⚠️ **Which is right depends on *why* the value is missing.** Filling every gap with `0` is easy, but a missing test score isn't the same as a score of zero, and it would drag the average down. Ask: is the value unknown, not applicable, or a data-entry error? Then fill it with something that makes sense (like the median), or drop the row.

### Fill or Drop? Some Rules of Thumb

| Situation | Often reasonable |
|---|---|
| The row's identifier (ID, name) is missing | Drop the row, or investigate the source |
| An optional field (a note, a comment) is missing | Keep the row and leave it missing |
| A category is missing, but the row is still useful | Fill with `'Unknown'` or `'Not Provided'` |
| A number is missing, and missing really means zero | Fill with `0` |
| A number is missing, and missing means *unknown* | Leave it, or fill with the median, and say so |
| Most of a column is missing | Consider dropping the column |
| *Whether* it was missing matters | Add a "was missing" flag column first |

You don't have to drop a row for **any** missing value. `subset=` drops only when **important** columns are missing, and `thresh=` keeps columns that have at least that many real values:

In [ ]:
print(df.dropna(subset=['Name']).shape)            # drop only rows with no Name
print(df.dropna(axis='columns', thresh=4).columns.tolist())   # keep columns with 4+ real values (all four qualify here)

### Keep a Record of What Was Missing
Filling hides the fact that a value was ever missing. If that might matter later, save a flag **first**:

In [ ]:
flags = df.copy()
flags['name_was_missing'] = flags['Name'].isnull()
flags['Name'] = flags['Name'].fillna('Not Provided')
flags[['Name', 'name_was_missing']]

### Filling From Neighbors: `ffill()` and `bfill()`
When rows are **in order**, like hourly readings, a gap is often best filled from the row before it (`ffill`, forward fill) or after it (`bfill`, backward fill). Don't use these when row order doesn't mean anything.

In [ ]:
temps = pd.Series([71.0, None, None, 74.5, None, 73.0],
                  index=['1pm', '2pm', '3pm', '4pm', '5pm', '6pm'])
pd.DataFrame({'raw': temps, 'ffill': temps.ffill(), 'bfill': temps.bfill()})

## 4. Data Type Fixes

In [ ]:
df.dtypes

Before converting, keep a copy of the raw values, so you can see afterwards exactly which ones **failed**.

⚠️ **The mixed-date trap.** Since pandas 2.0, `pd.to_datetime` guesses the format from the **first** value and turns everything that doesn't match it into `NaT`, **silently**, with no error and no warning. `Signup Date` mixes three formats, so tell pandas with `format='mixed'`:

In [ ]:
dates = df['Signup Date']
pd.DataFrame({
    'raw': dates,
    'guessed format': pd.to_datetime(dates, errors='coerce'),
    "format='mixed'": pd.to_datetime(dates, errors='coerce', format='mixed'),
})

Without `format='mixed'`, two perfectly good dates are lost. When **all** values share one known format, it's even better to state it exactly. For example, `format='%Y-%m-%d'` for `2026-01-05`, or `format='%m/%d/%Y'` for `01/05/2026` (`%Y` = 4-digit year, `%m` = month, `%d` = day).

In [ ]:
raw = df.copy()   # the original values, to check what failed

df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
df['Signup Date'] = pd.to_datetime(df['Signup Date'], errors='coerce', format='mixed')
df.dtypes

`errors='coerce'` turns anything it can't convert into `NaN` (or `NaT` for dates) instead of stopping with an error. Now find the values that **failed**: they're missing now, but weren't before:

In [ ]:
for col in ['Age', 'Score', 'Signup Date']:
    failed = raw[col][df[col].isna() & raw[col].notna()]
    print(f'{col:12} failed to convert: {failed.tolist()}')

`'thirty'`, `''`, and `'not a date'` are exactly the values a person would need to look at.

### Type Mismatches
A **type mismatch** is a value stored as the wrong type for what it means:

| Meant to be | Stored as | Problem |
|---|---|---|
| A number | `'19.99'` (text) | Can't be summed or averaged |
| A date | `'01/08/2026'` (text) | Sorts and filters as text, not by date |
| True/False | `'yes'`, `'Y'`, `'no'` | Several spellings of one idea |
| A category | `'IL'`, `'il'`, `'IL '` | One real category counted as three |
| Missing | `''`, `'N/A'`, `-1` | Fake values distort counts and averages |

### Nullable Types and `.astype()`
`.astype()` converts a column that's **already clean** to another type. One catch: a regular integer column (`int64`) **can't hold missing values**, so `.astype('int64')` fails if any are present. Pandas' **nullable** integer type, `'Int64'` (capital **I**), can:

In [ ]:
counts = pd.Series([2, None, 1, 3])
print(counts.dtype)                # float64: the None forced decimals
print(counts.astype('Int64'))      # whole numbers again, with <NA> for missing

| Type | Holds missing values? |
|---|---|
| `int64` | ❌ No, so `.astype('int64')` fails with `IntCastingNaNError` |
| `Int64` | ✅ Yes, as `<NA>` |
| `float64` / `Float64` | ✅ Yes |
| `'category'` | ✅ Yes, and saves memory for repeated text (Foundations notebook 08) |

## 5. Impute (Fill In) the Now-Numeric Missing Values

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Score'] = df['Score'].fillna(df['Score'].mean())
df['Signup Date'] = df['Signup Date'].fillna(df['Signup Date'].min())
df['Name'] = df['Name'].fillna('Unknown')

## 6. Cleaned Data Review

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 7. Duplicate Rows
The same record entered twice is a common problem, and it quietly inflates counts and totals.

In [ ]:
orders = pd.DataFrame({
    'order_id': [101, 102, 102, 103, 104, 104],
    'customer': ['Ana', 'Ben', 'Ben', 'Cy', 'Deb', 'Deb'],
    'amount':   [25.0, 40.0, 40.0, 15.5, 60.0, 65.0],
})

print(orders.duplicated())                 # True for each repeat of an earlier row
print('Exact duplicates:', orders.duplicated().sum())

In [ ]:
orders.drop_duplicates()

Order 104 appears twice with **different** amounts, so it isn't an exact duplicate. To treat rows as duplicates when only *some* columns match, use `subset=`. Use `keep='last'` to keep the most recent copy instead of the first:

In [ ]:
orders.drop_duplicates(subset=['order_id'], keep='last')

`orders.duplicated()` only marks the **second** copy. To see **every** member of a duplicate group side by side, use `keep=False`. This is the best way to decide what the duplicates actually are before deleting anything:

In [ ]:
orders[orders.duplicated(subset=['order_id'], keep=False)].sort_values('order_id')

Order 102 is the same purchase entered twice, so it's safe to drop one copy. Order 104 has two **different** amounts. Is it a correction, a second purchase, or an error? **Don't delete duplicates until you understand what they mean.** When the rows have a date, sort by it first, then `keep='last'` keeps the most recent version.

## 8. Replacing Placeholder Values
Datasets often mark missing data with a **code** instead of leaving it blank, such as `-1`, `999`, or `'N/A'`. Pandas sees those as real values, so they hide from `.isnull()` and distort the math. `.replace()` turns them into real `NaN`s:

In [ ]:
survey = pd.DataFrame({
    'age':    [34, -1, 27, 999, 45],
    'answer': ['yes', 'N/A', 'no', 'yes', 'N/A'],
})
print('Mean age before:', survey['age'].mean())

survey = survey.replace({-1: np.nan, 999: np.nan, 'N/A': np.nan})
print('Mean age after: ', survey['age'].mean())
survey

Replacing a value **everywhere** can be too blunt. `-1` might be a placeholder in one column but a real value in another. A **nested** dictionary limits each replacement to one column: `{column: {old: new}}`:

In [ ]:
readings = pd.DataFrame({
    'temp_change': [-1, 2, -3],          # -1 is a real value here
    'sensor_id':   [-1, 17, 22],         # ...but means "unknown" here
})
readings.replace({'sensor_id': {-1: np.nan}})

💡 If you know a file's placeholders up front, handle them **while loading**: `pd.read_csv('file.csv', na_values=['N/A', 'Unknown', -1])` (Foundations notebook 06). Be careful with `0`, which is usually a **real** value (zero items ordered), not a missing one.

## 9. Cleaning Text Categories
To a computer, `'IL'`, `'il'`, and `'IL '` are three **different** values. Standardize whitespace and capitalization first, then map alternate spellings to one value:

In [ ]:
states = pd.Series(['IL', 'il', ' IL ', 'Illinois', 'WI', 'wisconsin', 'WI'])
print(states.value_counts().to_dict())       # 6 "different" states!

clean = states.str.strip().str.upper().replace({'ILLINOIS': 'IL', 'WISCONSIN': 'WI'})
print(clean.value_counts().to_dict())        # really just 2

For **yes/no** answers, `.map()` with a dictionary is safer than `.replace()`. Anything the dictionary doesn't cover becomes `NaN`, so unexpected values **show up** instead of slipping through unchanged:

In [ ]:
answers = pd.Series([' Yes', 'y', 'NO', 'n ', 'maybe'])
lookup = {'yes': True, 'y': True, 'no': False, 'n': False}

tidy = answers.str.strip().str.lower()
print('replace:', tidy.replace(lookup).tolist())   # 'maybe' quietly survives
print('map:    ', tidy.map(lookup).tolist())       # 'maybe' becomes NaN, so you'll notice

## 10. Putting It Together: A Cleaning Pipeline
Here's a messier dataset with every problem from this notebook: `None`, empty strings, text and number placeholders, numbers stored as text, mixed date formats, stray spaces, and a duplicate row.

In [ ]:
sales = pd.DataFrame({
    'purchase_id':   [101, 102, 103, 104, 104, 105],
    'customer':      ['Avery', 'Jordan', None, 'Riley', 'Riley', ''],
    'amount':        ['19.99', 'N/A', '-1', '42.50', '42.50', ' 15.00 '],
    'quantity':      [2, None, 1, 3, 3, -1],
    'purchase_date': ['2026-01-05', '01/08/2026', 'not recorded', '2026-01-12', '2026-01-12', ''],
    'state':         ['IL', 'il ', 'Unknown', 'WI', 'WI', 'N/A'],
})
sales

A good pipeline works on a **copy**, so the raw data stays untouched for comparison, and it goes in a sensible order: **standardize the missing values first**, then clean text, then convert types, then remove duplicates and unusable rows.

In [ ]:
clean_sales = sales.copy()

# 1. Turn placeholders into real missing values, column by column
clean_sales = clean_sales.replace({
    'customer':      {'': pd.NA},
    'amount':        {'N/A': pd.NA, '-1': pd.NA},
    'quantity':      {-1: pd.NA},
    'purchase_date': {'not recorded': pd.NA, '': pd.NA},
    'state':         {'Unknown': pd.NA, 'N/A': pd.NA},
})

# 2. Clean text
clean_sales['state'] = clean_sales['state'].str.strip().str.upper()

# 3. Convert types
clean_sales['amount'] = pd.to_numeric(clean_sales['amount'], errors='coerce')
clean_sales['quantity'] = clean_sales['quantity'].astype('Int64')
clean_sales['purchase_date'] = pd.to_datetime(clean_sales['purchase_date'], errors='coerce', format='mixed')

# 4. Remove exact duplicates, then rows missing the fields a report needs
clean_sales = clean_sales.drop_duplicates()
clean_sales = clean_sales.dropna(subset=['purchase_id', 'amount', 'purchase_date'])

clean_sales

## 11. Validate After Cleaning
Cleaning isn't done until you've **checked** it. Compare before and after:

In [ ]:
print('shape:     ', sales.shape, '→', clean_sales.shape)
print('duplicates:', sales.duplicated().sum(), '→', clean_sales.duplicated().sum())
print('repeated IDs after:', clean_sales['purchase_id'].duplicated().sum())
print()
print(pd.DataFrame({'dtype before': sales.dtypes, 'dtype after': clean_sales.dtypes,
                    'missing before': sales.isna().sum(), 'missing after': clean_sales.isna().sum()}))

In [ ]:
print(clean_sales['amount'].describe())
print('Dates from', clean_sales['purchase_date'].min().date(), 'to', clean_sales['purchase_date'].max().date())
clean_sales['state'].value_counts(dropna=False)

Look at the **missing before** column: it found only 2 gaps. The placeholders (`'N/A'`, `-1`, `'Unknown'`, `''`) were invisible to `.isna()` until step 1 turned them into real missing values. That's why the pipeline standardizes missing values **first**.

Then ask yourself:
- Did any **valid** rows disappear? (Here, 104's duplicate, plus 102, 103, and 105, which had no usable amount or date. Is that what the report needs?)
- Did every fake value become a **real** missing value?
- Did the number and date columns convert, and what **failed**?
- Are impossible values still there (negative amounts, future dates)?
- Did duplicate removal use the right rule?

> Cleaning isn't just deleting inconvenient values. It's deciding which values are **valid**, which are **unknown**, and which need **investigating** before you can trust the analysis.

## 🏋️ Practice

### Practice 1 — Count and Compare (easy)
Before running any cleaning steps, how many total missing values were in the *original* `data` dictionary above (count them by hand, then verify with `.isnull().sum().sum()` on a fresh copy)?

In [ ]:
original = pd.DataFrame(data)
# Your code here


### Practice 2 — Real Missing Data (medium)
Load `ufo.csv` (in this folder) and print how many missing values each column has. Which column is the messiest?

In [ ]:
ufo = pd.read_csv('ufo.csv')
# Your code here


### Practice 3 — Fix a Type Column (harder)
In `ufo.csv`, the `date_time` column is stored as text, not an actual date. Convert it with `pd.to_datetime(..., errors='coerce')`, then check how many rows failed to convert (became `NaT`) with `.isnull().sum()`.

In [ ]:
# Your code here


### Practice 4 — Duplicate Names (medium)
`students.csv` has no exact duplicate rows, but some students share a **first** name. Use `duplicated(subset=['first_name'])` to list every repeated first name. Then make a version with only one student per first name.

In [ ]:
students = pd.read_csv('students.csv')
# Your code here


### Practice 5 — Standardize and Convert (medium)
1. Standardize this `state` Series so it contains only `'IL'` and `'WI'`, then show its `value_counts()`.
2. Convert `qty` to the nullable `'Int64'` type after turning its `-1` placeholder into a real missing value. Confirm the result holds `<NA>`.

In [ ]:
state = pd.Series(['il', 'IL ', 'Illinois', 'WI', ' wi', 'Wisconsin'])
qty = pd.Series([4, -1, 2, 7, -1])
# Your code here


## ✏️ Your Turn
Modify the `data` dictionary at the top of this notebook. Add:
- A new column with some `None` and `""` values
- At least one row with every column filled in incorrectly
- A placeholder value (like `'N/A'` or `-1`) that `.isnull()` won't catch

Then re-run the notebook and fix it step by step. Use `raw` (section 4) to show which values failed to convert, and compare the before and after missing-value counts.

In [ ]:
# Your turn here


## 🔥 Challenge (Optional) — Clean Real Data End-to-End
Load `ufo.csv` and build a cleaning **pipeline** like section 10:
- Identify which columns have missing values, and inspect the messiest one with `value_counts(dropna=False)`
- Use a mix of `.replace()`, `.fillna()`, `.dropna(subset=...)`, `pd.to_numeric()`, and `pd.to_datetime()` to clean it. Keep the raw data in a separate variable.
- **Validate** like section 11: compare `shape`, `dtypes`, missing-value counts, and duplicate counts before and after, and show which `date_time` values failed to convert. (*Hint:* look at their times.)

In [ ]:
# Your code here


---
### 📝 Summary
| Concept        | Tool/Function                          |
|----------------|------------------------------------------|
| Detect nulls   | `df.isnull()`, `df.isnull().sum()`, `.sum().sum()` |
| Inspect a column | `.value_counts(dropna=False)`, `.map(type).value_counts()` |
| Drop rows      | `df.dropna()`, `df.dropna(subset=[...])`  |
| Drop sparse columns | `df.dropna(axis='columns', thresh=n)` |
| Fill values    | `df.fillna()`, `.ffill()`, `.bfill()`     |
| Flag missing   | `df['x_was_missing'] = df['x'].isnull()`  |
| Convert types  | `pd.to_numeric(errors='coerce')`, `pd.to_datetime(errors='coerce', format='mixed')` |
| Nullable integers | `.astype('Int64')`                     |
| Replace values | `df.replace()`, nested `{col: {old: new}}` |
| Clean text     | `.str.strip().str.upper()`, `.map(lookup)` |
| Find duplicates | `df.duplicated()`, `.sum()`, `keep=False` to see all copies |
| Drop duplicates | `df.drop_duplicates(subset=..., keep=...)` |